# 04 — ViT / Frequency-Hybrid: Training Results & Cross-Generator Comparison
**Owner: Member C**

This notebook has two jobs:

1. Inspect and visualize **my own** Model 4 (ViT-B/16, optionally with the frequency-hybrid branch) — training curves, ROC curve, confusion matrices, primary vs. cross-generator performance.
2. Build the **final 4-model comparison** required by the report (Section 7 — Results & Model Comparison, Section 8 — Critical Analysis), by reading everyone's `results/<model>/.../metrics.csv` **read-only**. I never write into another member's `results/` folder — only into `results/comparison/`, which I own.

> Run each model's `train.py` first (see each model's own README) so the `results/` folders below are populated. Cells are written to fail gracefully with a warning if a file isn't there yet, so this notebook can be run early / iteratively.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

REPO_ROOT = Path("..")  # notebook lives in notebooks/, repo root is one level up
RESULTS = REPO_ROOT / "results"
COMPARISON_DIR = RESULTS / "comparison"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)


## 1. Config — where each model's results live

Read-only paths into the other members' folders. I don't edit anything inside them.

In [ ]:
# Map: display name -> (results folder, default run name used for the "headline" run)
# Update run_name values to match whichever run each member wants reported as final.
MODEL_RESULTS = {
    "Custom CNN":       {"dir": RESULTS / "custom_cnn",      "run_name": "custom_cnn_v1"},
    "ResNet50":         {"dir": RESULTS / "resnet50",        "run_name": "resnet50_v1"},
    "EfficientNetV2":   {"dir": RESULTS / "efficientnetv2",  "run_name": "efficientnetv2_v1"},
    "ViT-B/16":         {"dir": RESULTS / "vit",             "run_name": "vit_b16_baseline_v1"},
    # Optional ablation run — leave here if/when the frequency-hybrid run exists
    "ViT + Freq-Hybrid": {"dir": RESULTS / "vit",            "run_name": "vit_b16_freq_hybrid_v1"},
}


## 2. Load `metrics.csv` for every model (read-only)

Every model's `train.py` writes a `metrics.csv` with the same schema, so these can be concatenated directly into one comparison table.

In [ ]:
def load_metrics(model_name: str, model_dir: Path, run_name: str) -> pd.DataFrame | None:
    path = model_dir / run_name / "metrics.csv"
    if not path.exists():
        warnings.warn(f"[{model_name}] metrics.csv not found yet at {path} — run its train.py first.")
        return None
    df = pd.read_csv(path)
    df.insert(0, "display_name", model_name)
    return df


all_rows = []
for name, info in MODEL_RESULTS.items():
    df = load_metrics(name, info["dir"], info["run_name"])
    if df is not None:
        all_rows.append(df)

if all_rows:
    comparison_df = pd.concat(all_rows, ignore_index=True)
else:
    comparison_df = pd.DataFrame()

comparison_df


## 3. Final comparison table (Report Section 7)

Accuracy / Precision / Recall / F1 / ROC-AUC on the **primary test set** vs. the **cross-generator test set**, plus the accuracy drop — the single clearest evidence of generalization strength.

In [ ]:
display_cols = [
    "display_name", "test_acc", "test_precision", "test_recall", "test_f1", "test_auc",
    "cross_gen_acc", "cross_gen_precision", "cross_gen_recall", "cross_gen_f1", "cross_gen_auc",
    "accuracy_drop",
]

if not comparison_df.empty:
    report_table = comparison_df[[c for c in display_cols if c in comparison_df.columns]].copy()
    report_table = report_table.sort_values("accuracy_drop")
    display(report_table)

    # Save the report-ready table — this is the shared output folder I own.
    report_table.to_csv(COMPARISON_DIR / "final_comparison_table.csv", index=False)
    print(f"Saved -> {COMPARISON_DIR / 'final_comparison_table.csv'}")
else:
    print("No metrics loaded yet — nothing to save.")


## 4. Headline result — accuracy drop per model (generalization gap)

In [ ]:
if not comparison_df.empty:
    plot_df = comparison_df.sort_values("accuracy_drop")
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(plot_df["display_name"], plot_df["accuracy_drop"], color=sns.color_palette("crest", len(plot_df)))
    ax.set_ylabel("Accuracy drop (primary test → cross-generator test)")
    ax.set_title("Generalization gap across architectures\n(lower = generalizes better to unseen generator)")
    ax.bar_label(bars, fmt="%.3f")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / "accuracy_drop_comparison.png", dpi=150)
    plt.show()


## 5. Primary test vs. cross-generator accuracy — grouped bars

In [ ]:
if not comparison_df.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(comparison_df))
    width = 0.35
    ax.bar(x - width / 2, comparison_df["test_acc"], width, label="Primary test (in-distribution)")
    ax.bar(x + width / 2, comparison_df["cross_gen_acc"], width, label="Cross-generator (unseen)")
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_df["display_name"], rotation=20, ha="right")
    ax.set_ylabel("Accuracy")
    ax.set_title("In-distribution vs. cross-generator accuracy")
    ax.legend()
    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / "indist_vs_crossgen_accuracy.png", dpi=150)
    plt.show()


## 6. ROC curves — all models, primary test set and cross-generator set

Reads each model's `roc_primary_test.csv` / `roc_cross_gen.csv` (same schema every model's `train.py` writes: `fpr`, `tpr`).

In [ ]:
def plot_roc_overlay(tag: str, title: str, save_name: str):
    fig, ax = plt.subplots(figsize=(6.5, 6))
    for name, info in MODEL_RESULTS.items():
        roc_path = info["dir"] / info["run_name"] / f"roc_{tag}.csv"
        if not roc_path.exists():
            continue
        roc_df = pd.read_csv(roc_path)
        ax.plot(roc_df["fpr"], roc_df["tpr"], label=name)
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1, label="Chance")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / save_name, dpi=150)
    plt.show()


plot_roc_overlay("primary_test", "ROC — Primary (in-distribution) test set", "roc_overlay_primary_test.png")
plot_roc_overlay("cross_gen", "ROC — Cross-generator (unseen) test set", "roc_overlay_cross_gen.png")


## 7. Confusion matrices — side by side (primary test vs. cross-generator), per model

In [ ]:
def load_cm(model_dir: Path, run_name: str, tag: str):
    path = model_dir / run_name / f"confusion_matrix_{tag}.csv"
    if not path.exists():
        return None
    return np.loadtxt(path, delimiter=",")


active_models = {n: i for n, i in MODEL_RESULTS.items()
                  if load_cm(i["dir"], i["run_name"], "primary_test") is not None}

if active_models:
    n = len(active_models)
    fig, axes = plt.subplots(n, 2, figsize=(7, 3.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)

    for row, (name, info) in enumerate(active_models.items()):
        for col, tag in enumerate(["primary_test", "cross_gen"]):
            cm = load_cm(info["dir"], info["run_name"], tag)
            ax = axes[row, col]
            if cm is None:
                ax.axis("off")
                continue
            sns.heatmap(cm, annot=True, fmt=".0f", cbar=False, cmap="Blues",
                        xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"], ax=ax)
            ax.set_title(f"{name} — {'Primary test' if tag == 'primary_test' else 'Cross-gen'}")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / "confusion_matrices_grid.png", dpi=150)
    plt.show()
else:
    print("No confusion matrices found yet.")


## 8. Computational efficiency vs. accuracy (Report Section 6 / rubric requirement)

Training time, inference time per image, and parameter count, plotted against cross-generator accuracy — the trade-off that matters most for a real deployment decision.

In [ ]:
if not comparison_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    sc = axes[0].scatter(
        comparison_df["inference_time_sec_per_image"] * 1000,
        comparison_df["cross_gen_acc"],
        s=comparison_df["num_params"] / comparison_df["num_params"].max() * 800 + 50,
        c=range(len(comparison_df)),
        cmap="viridis",
        alpha=0.8,
    )
    for _, r in comparison_df.iterrows():
        axes[0].annotate(r["display_name"], (r["inference_time_sec_per_image"] * 1000, r["cross_gen_acc"]),
                          textcoords="offset points", xytext=(6, 4), fontsize=8)
    axes[0].set_xlabel("Inference time (ms / image)")
    axes[0].set_ylabel("Cross-generator accuracy")
    axes[0].set_title("Accuracy vs. inference speed\n(bubble size = parameter count)")

    axes[1].bar(comparison_df["display_name"], comparison_df["num_params"] / 1e6,
                color=sns.color_palette("crest", len(comparison_df)))
    axes[1].set_ylabel("Parameters (millions)")
    axes[1].set_title("Model size comparison")
    axes[1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / "efficiency_vs_accuracy.png", dpi=150)
    plt.show()


## 9. Frequency-hybrid ablation — does the FFT branch help cross-generator generalization?

Compares the two ViT rows (`use_frequency_hybrid=false` vs. `true`) directly, if both runs exist.

In [ ]:
vit_rows = comparison_df[comparison_df["display_name"].str.contains("ViT")] if not comparison_df.empty else pd.DataFrame()

if len(vit_rows) >= 2:
    display(vit_rows[["display_name", "test_acc", "cross_gen_acc", "accuracy_drop", "num_params"]])
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.bar(vit_rows["display_name"], vit_rows["cross_gen_acc"], color=["#4C72B0", "#DD8452"])
    ax.set_ylabel("Cross-generator accuracy")
    ax.set_title("Frequency-hybrid branch: cross-generator accuracy")
    plt.xticks(rotation=10)
    plt.tight_layout()
    plt.savefig(COMPARISON_DIR / "freq_hybrid_ablation.png", dpi=150)
    plt.show()
else:
    print("Need both the baseline ViT run and the frequency-hybrid run to compare "
          "(train configs/vit.yaml with use_frequency_hybrid: true/false, different run_names).")


## 10. Notes for the report (fill in after runs complete)

- **Which model generalized best** (lowest `accuracy_drop`) and a hypothesis for *why* (e.g. inductive bias, pretraining, frequency-domain signal).
- **Which model was most efficient** (accuracy vs. inference time / parameter count trade-off from Section 8).
- **Whether the frequency-hybrid branch helped**, and by how much, from Section 9.
- Cross-check `final_comparison_table.csv` numbers against each model's own notebook (`02`, `03`) before pasting into the report — this notebook only aggregates, it doesn't re-run anyone else's evaluation.